In [ ]:
import pandas as pd
import numpy as np
from functools import reduce

ID_COL = "id"
PRED_COL = "Will_Buy_EV"

def load_preds_table(paths, id_col=ID_COL, pred_col=PRED_COL, prefix="S"):
    dfs = []
    for i, p in enumerate(paths):
        df = pd.read_csv(p)[[id_col, pred_col]].rename(columns={pred_col: f"{prefix}{i}"})
        dfs.append(df)
    m = reduce(lambda l, r: l.merge(r, on=id_col, how="inner"), dfs)
    return m

def power_mean_blend(paths, weights=None, p=8, clip_eps=1e-12):
    m = load_preds_table(paths, prefix="S")
    cols = [c for c in m.columns if c != ID_COL]
    P = m[cols].to_numpy(float)

    P = np.clip(P, clip_eps, 1.0 - clip_eps)

    k = P.shape[1]
    if weights is None:
        w = np.ones(k, dtype=float) / k
    else:
        w = np.array(weights, dtype=float)
        w = w / (w.sum() + 1e-12)

    # generalized mean (power mean)
    blend = (P ** p) @ w
    blend = np.clip(blend, clip_eps, None)
    blend = np.power(blend, 1.0 / p)

    out = pd.DataFrame({ID_COL: m[ID_COL].values, PRED_COL: blend})
    return out

paths = [
    "../input/sample_submission.csv",
    "../input/0.94621.csv",
    "../input/0.94619.csv",
]

# weights = [2.99, 0.1, 0.1, 0.1, 0.1, -0.2]
weights = [1.0, 1.0, 1.0]

final = power_mean_blend(paths, weights=weights, p=16)
final.to_csv("submission.csv", index=False)
print("saved submission.csv | rows:", len(final))


✅ saved submission.csv | rows: 286571
